# Analyse des métriques KGE / NSE — BFU_LSTM (ovk001)

Ce notebook lit les fichiers `results/{bassin}.csv` produits par `evaluate_ensemble_new.ipynb` (colonnes `qobs_mmd`, `qsim_mmd`, `qobs_m3s`, `qsim_m3s`, indexées par date), calcule le **KGE** et le **NSE** par bassin, et sépare les résultats en deux périodes :

- **Calibration** : `train_start_date` → `train_end_date` de votre `config.yml` (la période utilisée à l'entraînement)
- **Validation temporelle** : `test_start_date` → `test_end_date` (période jamais vue à l'entraînement, mêmes bassins)

Adapté à partir de trois versions précédentes trouvées dans ce dossier (`metrics_analysis.ipynb`, `metrics_analysis_marie.ipynb`, `metrics_analysis_spatiotemp_val.ipynb`). Voir la dernière cellule pour ce qui a changé et pourquoi.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm
from IPython.display import display

import neuralhydrology.evaluation.metrics as nhm

## Configuration — à adapter si votre projet ou vos dates changent

In [ ]:
# Dossier où evaluate_ensemble_new.ipynb a écrit un .csv par bassin
# (qobs_mmd, qsim_mmd, qobs_m3s, qsim_m3s, indexé par date)
RESULTS_DIR = Path('/home/ovk001/store8/BFU_LSTM/new_basin/EXPERIMENT/Apply_BFU_Experiment_Exp1/results')

# Liste des bassins à évaluer. Dans ce projet, train_basin_file == validation_basin_file
# == test_basin_file (voir config.yml de vos runs) : tous les bassins servent à la fois à
# l'entraînement et au test. Il n'y a donc pas de bassins mis de côté pour une "validation
# spatiale" — voir la note en bas de notebook si vous voulez en construire une un jour.
BASIN_LIST_PATH = Path('/home/ovk001/store8/BFU_LSTM/new_basin/EXPERIMENT/BFU_LSTM_Exp1/basins/all_available_basins_list.txt')

# Coupures de date — doivent correspondre à train_start_date / train_end_date / test_start_date
# / test_end_date de votre config.yml (copiez les valeurs exactes depuis un config.yml de run).
CALIBRATION_START = '2010-01-01'
CALIBRATION_END = '2021-01-01'    # exclusif : la calibration s'arrête juste avant test_start_date
VALIDATION_START = '2021-01-01'
VALIDATION_END = '2025-01-01'     # exclusif : mettez une date après votre test_end_date réel

METRICS_OUT_DIR = RESULTS_DIR     # où écrire metrics_cal.csv / metrics_tmp.csv

## Calcul des métriques par bassin

`nhm.kge` et `nhm.nse` acceptent directement des colonnes `pandas.Series` (pas besoin de les convertir en `xarray.DataArray` — les deux fonctions ne font que des opérations qui marchent sur les deux types, comme `.isnull()` et `.shape`).

In [ ]:
def compute_metrics_per_basin(basin_list_path: Path, results_dir: Path,
                              period_start: str, period_end: str) -> pd.DataFrame:
    """Calcule KGE et NSE pour chaque bassin de `basin_list_path`, sur [period_start, period_end).

    Lit `results_dir/{basin}.csv` (produit par evaluate_ensemble_new.ipynb) pour chaque bassin.
    Un bassin sans fichier résultat est ignoré et listé à la fin (jamais évalué, ou nom de
    bassin qui ne correspond à aucun fichier .csv).
    """
    with open(basin_list_path) as f:
        basins = [line.strip() for line in f if line.strip()]

    rows = []
    missing_basins = []
    for basin in tqdm(basins):
        result_file = results_dir / f'{basin}.csv'
        if not result_file.is_file():
            missing_basins.append(basin)
            continue

        df = pd.read_csv(result_file, index_col=0)
        df_period = df[(df.index >= period_start) & (df.index < period_end)]

        rows.append({
            'basin': basin,
            'KGE': nhm.kge(df_period['qobs_mmd'], df_period['qsim_mmd']),
            'NSE': nhm.nse(df_period['qobs_mmd'], df_period['qsim_mmd']),
        })

    if missing_basins:
        print(f'{len(missing_basins)} bassin(s) sans fichier résultat (ignorés) : {missing_basins}')

    return pd.DataFrame(rows)

In [ ]:
metrics_cal = compute_metrics_per_basin(BASIN_LIST_PATH, RESULTS_DIR, CALIBRATION_START, CALIBRATION_END)
metrics_tmp = compute_metrics_per_basin(BASIN_LIST_PATH, RESULTS_DIR, VALIDATION_START, VALIDATION_END)

metrics_cal.to_csv(METRICS_OUT_DIR / 'metrics_cal.csv', index=False)
metrics_tmp.to_csv(METRICS_OUT_DIR / 'metrics_tmp.csv', index=False)

print(f'Calibration            : {len(metrics_cal)} bassins')
print(f'Validation temporelle  : {len(metrics_tmp)} bassins')

## Boxplots KGE et NSE

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(10, 5))

labels = [f'Calibration\n({len(metrics_cal)} bassins)', f'Validation temp.\n({len(metrics_tmp)} bassins)']

axs[0].boxplot([metrics_cal['KGE'].dropna(), metrics_tmp['KGE'].dropna()], showfliers=False)
axs[0].set_xticklabels(labels)
axs[0].set_title('KGE')

axs[1].boxplot([metrics_cal['NSE'].dropna(), metrics_tmp['NSE'].dropna()], showfliers=False)
axs[1].set_xticklabels(labels)
axs[1].set_title('NSE')

plt.tight_layout()
plt.savefig(RESULTS_DIR / 'boxplot_kge_nse.png')
plt.show()

## Bassins hors norme (outliers)

Méthode de Tukey : un bassin est "hors norme" si sa métrique tombe sous la moustache basse d'un boxplot (Q1 − 1.5×IQR, ajusté à la valeur réelle la plus basse au-dessus de ce seuil).

In [ ]:
def find_outliers(df_metric: pd.DataFrame, metric: str, whis: float = 1.5) -> pd.DataFrame:
    """Retourne les bassins dont `metric` est sous la moustache basse d'un boxplot (méthode de Tukey)."""
    d = df_metric[metric].dropna()
    q1, q3 = np.quantile(d, [0.25, 0.75])
    iqr = q3 - q1
    low_bound = q1 - whis * iqr

    above_bound = d[d >= low_bound]
    wisk_lo = above_bound.min() if len(above_bound) else q1

    print(f'{metric} — médiane: {d.median():.3f}  écart-type: {d.std():.3f}  moustache basse: {wisk_lo:.3f}')

    return df_metric[df_metric[metric] < wisk_lo][['basin', metric]]

In [ ]:
print('--- Calibration ---')
display(find_outliers(metrics_cal, 'KGE'))
display(find_outliers(metrics_cal, 'NSE'))

print('--- Validation temporelle ---')
display(find_outliers(metrics_tmp, 'KGE'))
display(find_outliers(metrics_tmp, 'NSE'))

## (Optionnel) Vérifier pourquoi les résultats s'arrêtent au 31 décembre 2023

Ce n'est pas ce notebook qui limite la date, ni `evaluate_ensemble_new.ipynb` — c'est probablement la donnée d'observation ou de forçage sous-jacente qui ne va pas plus loin, même si `test_end_date: 31/12/2024` est demandé dans le config. Vérifiez la période réellement couverte par un de vos fichiers NetCDF :

In [ ]:
# import xarray as xr
# ds = xr.open_dataset('/chemin/vers/un/fichier_de_donnees.nc')
# print(ds.time.min().values, ds.time.max().values)
# Si ça confirme que la donnée s'arrête au 31/12/2023, le modèle ne peut simplement pas
# produire de prédictions au-delà — ce n'est pas un bug de ce pipeline.

## Notes — ce qui a changé par rapport aux versions précédentes

1. **NSE ajouté.** Les 3 anciennes versions ne calculaient que le KGE, alors que `metrics: ['NSE', 'KGE']` est défini dans vos seed files. Le NSE est maintenant calculé partout où le KGE l'est.
2. **Seulement 2 groupes (calibration / validation temporelle), pas 4.** Les anciennes versions calculaient aussi une "validation spatiale" et une "validation spatio-temporelle", avec une liste de bassins distincte de celle du train. Dans votre `config.yml`, `train_basin_file` == `validation_basin_file` == `test_basin_file` : aucun bassin n'est mis de côté pour tester la généralisation spatiale. Recalculer une "validation spatiale" sur les mêmes bassins que l'entraînement ne mesurerait rien de plus que la validation temporelle elle-même — donc ce n'est pas fait ici.
   - Si vous voulez un jour une vraie validation spatiale : retirez un sous-ensemble de bassins de `train_basin_file` avant l'entraînement (pour qu'ils ne soient jamais vus par le modèle), et créez une liste séparée de ces bassins exclus pour ce notebook.
3. **Chemins adaptés à `ovk001`.** Les anciennes versions avaient des chemins codés en dur vers `/home/mba002/...` et `/home/ega001/...` — remplacés par votre propre chemin de projet (`Apply_BFU_Experiment_Exp1`), regroupés dans une seule cellule de configuration en haut du notebook.
4. **Cellules de debug retirées.** `metrics_analysis_marie.ipynb` contenait des cellules de comparaison ancien/nouveau modèle spécifiques au travail de Marie, dont une qui plantait (`NameError: metric_original`) et une avec un bug dans `pd.merge`. Pas reprises ici — dites-moi si vous avez besoin d'un notebook de comparaison équivalent pour votre propre projet.